# Full run: H-ETE-GNN vs HR-ETE-GNN

The controlled comparison at the thesis's full scale, configured to mimic Lee & Cho (2025) as closely as `../replicable.md` allows.

| | H-ETE-GNN (baseline) | HR-ETE-GNN (ours) |
|---|---|---|
| Edge matrix | Shannon ETE, α = 1 | Rényi ETE, α\* |
| Everything else | identical | identical |

"Everything else" means the same data, estimator (3-bin discretised TE, k = l = 1, m = 100 shuffled surrogates, Z > 1.96), architecture, training, Hurst regime calendar, seeds and surrogate permutations. At α = 1 the Rényi builder calls the baseline's own code, so the two arms are nested exactly.

**Cut to save compute:** only these two models are run, and only under the Hurst retraining policy. The periodic control and the TE / Granger / Pearson / LSTM / GRU benchmarks are dropped.

**Protocol**
1. **Stage 1, α selection.** Walk-forward over the validation span (forecast days up to `VAL_END`) for every α in the grid. α\* is the order with the lowest mean validation MSE, which is the training loss.
2. **Stage 2, full run.** Walk-forward over 2006–2024 for α = 1 and α = α\*, 10 seeds each.
3. **Confirmatory test.** Only forecast days after `VAL_END`, which α\* never saw. Giacomini–White on QLIKE is the test of record; Diebold–Mariano is reported as descriptive.

Every run is checkpointed to `runs/`. If the kernel dies, re-run the notebook and it carries on where it stopped.

## 0. Settings

In [ ]:
import os, sys, json, time
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

HERE = Path.cwd()
if HERE.name != "final_results":           # allow launching from the repo root
    HERE = HERE / "final_results"
sys.path.insert(0, str(HERE)); sys.path.insert(0, str(HERE.parent))

import torch
import hetegnn_base as hb
import hrete_full as hf
import thesis_stats as ts

# ---- run controls ----------------------------------------------------------
DEVICE = "auto"      # "auto" benchmarks cpu vs cuda in section 2 and picks the faster
N_JOBS = None        # parallel runs; None = chosen in section 2 from the core count
REFRESH_DATA = False # True re-downloads from Yahoo; False reuses data/panel.pkl

# Smoke mode: a few minutes, tiny budgets, separate output folder.  Only for
# checking the notebook runs end to end - its numbers mean nothing.
SMOKE = os.environ.get("RUN_FULL_SMOKE") == "1"

ALPHA_GRID, SEEDS, VAL_SEEDS = hf.ALPHA_GRID, hf.SEEDS, hf.VAL_SEEDS
RUNS_ROOT = HERE / "runs"
TABLES = HERE / "tables"
OVERRIDES = {}
if SMOKE:
    ALPHA_GRID, SEEDS, VAL_SEEDS = (0.7, 1.0), (0, 1), (0,)
    RUNS_ROOT, TABLES = HERE / "runs_smoke", HERE / "tables_smoke"
    OVERRIDES = dict(epochs=2, finetune_epochs=1, n_shuffles=5)
TABLES.mkdir(parents=True, exist_ok=True)

print(f"torch {torch.__version__}   cuda available: {torch.cuda.is_available()}"
      + (f" ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else ""))
print(f"CPU cores: {os.cpu_count()}   SMOKE={SMOKE}")

## 1. Configuration

Everything below comes from `hrete_full.make_config()`. Rows marked *gap* are values the paper does not report; the thesis fixes them (`replicable.md` §8) and applies them identically to both arms.

In [ ]:
cfg = hf.make_config(**OVERRIDES)
c = cfg.__dict__
rows = [
    ("Data", "Tickers", ", ".join(cfg.tickers), "Specified"),
    ("Data", "Regime proxy", " -> ".join(cfg.proxy_chain), "Gap 4: URTH spliced with ACWI, IOO"),
    ("Data", "Sample", f"{cfg.start} .. 2024-05-29", "Specified"),
    ("Data", "RV window", cfg.rv_window, "Specified"),
    ("TE", "Bins / lag", f"{cfg.n_bins} / {cfg.te_lag}", "Specified"),
    ("TE", "Surrogates m", cfg.n_shuffles, "Gap 2: fixed at 100"),
    ("TE", "Surrogate type", "shuffle (both arms)", "Specified; resolves replicable.md s10"),
    ("TE", "Edge rule", f"Z > {cfg.z_threshold}", "Specified"),
    ("TE", "Adjacency", "row-normalised" if cfg.normalize_adjacency else "raw", "Departure (s9)"),
    ("Model", "Conv kernels x channels", f"{cfg.conv_kernels} x {cfg.conv_channels}", "Specified"),
    ("Model", "Hidden sizes", f"{cfg.hidden1}, {cfg.hidden2}", "Gap 1: frozen, not tuned"),
    ("Train", "Optimiser / lr / batch", f"Adam / {cfg.lr} / {cfg.batch_size}", "Gap 1: frozen, not tuned"),
    ("Train", "Epochs / patience", f"{cfg.epochs} / {cfg.patience}", "Gap 3: full budget, best weights kept"),
    ("Train", "Train:val split", f"{1-cfg.val_fraction:.0%}:{cfg.val_fraction:.0%}", "Specified"),
    ("Train", "M / N", f"{cfg.input_size} / {cfg.train_period}", "Specified (Table 3)"),
    ("Train", "Scaler fit", "each N-day training block", "Gap 8"),
    ("Hurst", "Window / lookback / s", f"{cfg.hurst_window} / {cfg.regime_lookback} / {cfg.regime_sensitivity}", "Specified"),
    ("Hurst", "R/S correction", cfg.hurst_correction, "Gap 6: literal R/S"),
    ("Hurst", "Fine-tune epochs", cfg.finetune_epochs, "Gap 5"),
    ("Run", "alpha grid", ALPHA_GRID, "alpha* chosen on validation"),
    ("Run", "Validation span ends", hf.VAL_END, "alpha* never sees later days"),
    ("Run", "Seeds", f"{SEEDS} (both arms)", "Gap 11"),
]
pd.DataFrame(rows, columns=["Group", "Item", "Value", "Status"]).style.hide(axis="index")

## 2. Data

In [ ]:
data = hf.prepare_data(hf.make_config(), refresh=REFRESH_DATA)
prices, rv, changes = data["prices"], data["rv"], pd.DatetimeIndex(data["changes"])
print(f"prices   {prices.shape}  {prices.index[0].date()} -> {prices.index[-1].date()}")
print("proxy    " + ", ".join(f"{k}={v}" for k, v in data["proxy_src"].value_counts().items()))
print(f"hurst    mean={data['hurst'].mean():.3f}   above 0.5: {(data['hurst'] > 0.5).mean():.1%}")
print(f"regimes  {len(changes)} changes over the sample (paper reports 47)")

fig, ax = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
ax[0].plot(rv.mean(axis=1), lw=.8, color="#1a1a2e"); ax[0].set_ylabel("mean RV, %")
ax[1].plot(data["hurst"], lw=.8, color="#16a085"); ax[1].axhline(.5, color="k", lw=.6)
ax[1].set_ylabel("rolling H")
for a in ax:
    a.axvline(pd.Timestamp(hf.VAL_END), color="#e94560", ls="--", lw=1)
    for d in changes: a.axvline(d, color="#f39c12", lw=.4, alpha=.6)
ax[0].set_title("Realised volatility and the Hurst regime calendar (orange = regime change, red = end of validation)")
plt.tight_layout(); plt.show()

## 3. Device, parallelism and run-time estimate

This times one 100-epoch fit and one graph build on each available device, then projects the whole experiment from those times. The network is tiny and trains in batches of 8, so a GPU is **not** automatically faster: kernel-launch overhead can outweigh the compute. `DEVICE = "auto"` picks whichever device measured faster.

Runs are independent, so several run at once in separate processes. On CPU each process gets a share of the cores. On GPU the processes share the card; each uses only a few hundred MB.

In [ ]:
cands = ["cpu"] + (["cuda"] if torch.cuda.is_available() else [])
if DEVICE != "auto":
    cands = [DEVICE]

bench = {}
for dev in cands:
    b = hf.benchmark(hf.make_config(device=dev, torch_threads=2, **OVERRIDES), data)
    bench[dev] = b
    print(f"{dev:5s}  one full fit {b['fit_seconds']:6.1f} s   graph build "
          + ", ".join(f"a={a:g}: {t:.2f} s" for a, t in b["graph_seconds"].items()))
DEV = min(bench, key=lambda d: bench[d]["fit_seconds"])

cores = os.cpu_count() or 1
if N_JOBS is None:
    # cpu: 2 torch threads per run; cuda: graph building is CPU work, so one run per core-pair as well
    N_JOBS = max(1, min(8, cores // 2))
THREADS = max(1, min(4, cores // N_JOBS))
RUN_OVERRIDES = dict(OVERRIDES, device=DEV, torch_threads=THREADS)
print(f"\nusing device={DEV}  n_jobs={N_JOBS}  torch threads per run={THREADS}")

est = hf.project_runtime(bench[DEV], data, hf.make_config(**RUN_OVERRIDES),
                         alpha_grid=ALPHA_GRID, val_seeds=VAL_SEEDS, seeds=SEEDS, n_jobs=N_JOBS)
display(est.round(2))
print("Parallel speed-up is rarely perfect: expect the n_jobs column to be optimistic by ~20-40%.")

## 4. Stage 1: select α\* on the validation span

Each α runs on forecast days up to `VAL_END` with the validation seeds. α\* is the order with the lowest validation MSE, averaged over seeds. QLIKE is shown for information only and plays no part in the choice. α = 1 is in the grid, so if no Rényi order beats Shannon on validation, α\* = 1 and the test has nothing to compare. That outcome is reported as found.

In [ ]:
val_specs = [hf.RunSpec("val", a, s, str(RUNS_ROOT)) for a in ALPHA_GRID for s in VAL_SEEDS]
_ = hf.run_many(val_specs, n_jobs=N_JOBS, cfg_overrides=RUN_OVERRIDES)

In [ ]:
def qlike_mean(y, p):
    return float(ts.qlike(y, p, warn=False).mean())

rows = []
for s in val_specs:
    r = hf.load_run(s)
    rows.append(dict(alpha=s.alpha, seed=s.seed,
                     MSE=float(np.mean((r["y_pred"] - r["y_true"]) ** 2)),
                     QLIKE=qlike_mean(r["y_true"], r["y_pred"]), refits=r["n_refits"]))
val_tab = (pd.DataFrame(rows).groupby("alpha")
           .agg(MSE_mean=("MSE", "mean"), MSE_sd=("MSE", "std"),
                QLIKE_mean=("QLIKE", "mean"), refits=("refits", "first")))
ALPHA_STAR = float(val_tab["MSE_mean"].idxmin())
val_tab["selected"] = np.where(val_tab.index == ALPHA_STAR, "<- alpha*", "")
display(val_tab.round(4))
val_tab.to_csv(TABLES / "stage1_alpha_selection.csv")
json.dump({"alpha_star": ALPHA_STAR, "val_end": hf.VAL_END, "val_seeds": list(VAL_SEEDS)},
          open(TABLES / "alpha_star.json", "w"), indent=2)
print(f"alpha* = {ALPHA_STAR:g}   (frozen; written to {TABLES / 'alpha_star.json'})")

## 5. Stage 2: full walk-forward, both arms, 10 seeds

This is the long part; section 3 gives the estimate. Progress and an ETA print after each run. If α\* = 1 only the baseline runs.

In [ ]:
ARMS = sorted({1.0, ALPHA_STAR})
full_specs = [hf.RunSpec("full", a, s, str(RUNS_ROOT)) for a in ARMS for s in SEEDS]
t0 = time.time()
_ = hf.run_many(full_specs, n_jobs=N_JOBS, cfg_overrides=RUN_OVERRIDES)
print(f"stage 2 wall clock this session: {(time.time() - t0) / 3600:.2f} h")

## 6. Results

### 6.1 Replication audit: the paper's metrics over 2006–2024

Mean ± SD over the 10 seeds, laid out like the paper's Table 6, with the paper's Hurst / ETE-GNN row for reference.

In [ ]:
runs = {(s.alpha, s.seed): hf.load_run(s) for s in full_specs}
paper_cols = ["RMSE", "MAE", "MAPE", "Correlation", "HitRatio"]

rows = []
for (a, s), r in runs.items():
    m = hb.metrics(r["y_true"], r["y_pred"])
    m.update(Model=hf.arm_label(a), seed=s, QLIKE=qlike_mean(r["y_true"], r["y_pred"]),
             refits=r["n_refits"], minutes=r["seconds"] / 60)
    rows.append(m)
per_seed = pd.DataFrame(rows)
per_seed.to_csv(TABLES / "full_span_per_seed.csv", index=False)

agg = per_seed.groupby("Model")[paper_cols + ["QLIKE"]].agg(["mean", "std"])
fmt = pd.DataFrame({c: agg[(c, "mean")].map("{:.4f}".format) + " ± " + agg[(c, "std")].map("{:.4f}".format)
                    for c in paper_cols + ["QLIKE"]})
paper = hb.PAPER_TABLE6.query("Regime == 'Hurst' and Model == 'ETE-GNN'")[paper_cols]
fmt.loc["Lee & Cho (2025), H-ETE-GNN"] = paper.iloc[0].map("{:.4f}".format).tolist() + ["-"]
display(fmt)
fmt.to_csv(TABLES / "full_span_paper_metrics.csv")

### 6.2 Confirmatory test: forecast days after `VAL_END`

The forecast for each arm is the **mean over the 10 seeds**, which removes initialisation noise. The daily loss is QLIKE averaged over the 10 ETFs.

- **H1 (test of record):** unconditional Giacomini–White, d = QLIKE(H-ETE-GNN) − QLIKE(HR-ETE-GNN). The p-value printed is the GW Wald test, which is two-sided; with one restriction the one-sided p is half of it when the mean difference is positive.
- **Descriptive:** Diebold–Mariano with the HLN correction (one-sided). The arms are nested, so DM is undersized; it is not evidence.
- **Secondary:** conditional GW with the lagged Hurst regime as the instrument. This asks whether the regime predicts when Rényi wins.
- **Per ETF:** one-sided DM per ETF, with a Benjamini–Hochberg correction at q = 0.10.

In [ ]:
if ALPHA_STAR == 1.0:
    print("alpha* = 1: the selected model IS the baseline. There is no comparison to test.")
else:
    def ensemble(a):
        rs = [runs[(a, s)] for s in SEEDS]
        return rs[0]["dates"], rs[0]["y_true"], np.mean([r["y_pred"] for r in rs], axis=0)

    dates, y, p_base = ensemble(1.0)
    _, _, p_new = ensemble(ALPHA_STAR)
    test = dates > pd.Timestamp(hf.VAL_END)
    dt, yt, pb, pn = dates[test], y[test], p_base[test], p_new[test]
    print(f"test span {dt[0].date()} -> {dt[-1].date()}  ({test.sum()} days x {yt.shape[1]} ETFs)")
    for name, p in [("H-ETE-GNN", pb), (hf.arm_label(ALPHA_STAR), pn)]:
        print(f"  {name:22s} forecasts below the QLIKE floor: {(p < 0.05).sum()}")

    Lb, Ln = ts.qlike(yt, pb, warn=False), ts.qlike(yt, pn, warn=False)
    Sb, Sn = (pb - yt) ** 2, (pn - yt) ** 2
    lb, ln = Lb.mean(axis=1), Ln.mean(axis=1)

    gw = ts.giacomini_white(lb, ln)
    dm = ts.diebold_mariano(lb, ln, alternative="greater")
    dm_mse = ts.diebold_mariano(Sb.mean(axis=1), Sn.mean(axis=1), alternative="greater")
    lab = data["labels"].map({"Above": 1.0, "Below": 0.0}).shift(1).reindex(dt).ffill().bfill()
    gw_reg = ts.giacomini_white(lb, ln, instruments=lab.values)

    headline = pd.DataFrame([
        ("QLIKE, H-ETE-GNN", lb.mean()), (f"QLIKE, {hf.arm_label(ALPHA_STAR)}", ln.mean()),
        ("QLIKE improvement, %", dm["pct_improvement"]),
        ("GW unconditional: Wald stat", gw["stat"]), ("GW unconditional: p (two-sided)", gw["p_value"]),
        ("GW unconditional: p (one-sided)", gw["p_value"] / 2 if dm["mean_diff"] > 0 else 1 - gw["p_value"] / 2),
        ("DM (HLN) on QLIKE: stat", dm["stat"]), ("DM (HLN) on QLIKE: p one-sided", dm["p_value"]),
        ("MSE improvement, %", dm_mse["pct_improvement"]), ("DM on MSE: p one-sided", dm_mse["p_value"]),
        ("GW with lagged regime: Wald stat", gw_reg["stat"]), ("GW with lagged regime: p", gw_reg["p_value"]),
        ("GW regime coefficient", gw_reg["coef"].iloc[1]), ("GW regime coefficient p", gw_reg["p_coef"].iloc[1]),
    ], columns=["quantity", "value"]).set_index("quantity")
    display(headline.round(5))
    headline.to_csv(TABLES / "test_headline.csv")

    per_etf = []
    for k, t in enumerate(cfg.tickers):
        d = ts.diebold_mariano(Lb[:, k], Ln[:, k], alternative="greater")
        per_etf.append(dict(ETF=t, QLIKE_base=Lb[:, k].mean(), QLIKE_new=Ln[:, k].mean(),
                            improvement_pct=d["pct_improvement"], DM_stat=d["stat"], p=d["p_value"]))
    per_etf = pd.DataFrame(per_etf)
    bh = ts.benjamini_hochberg(per_etf["p"].values, q=0.10)
    per_etf = pd.concat([per_etf, bh.drop(columns=["p_value"])
                         .reset_index(drop=True)], axis=1)
    display(per_etf.round(4))
    per_etf.to_csv(TABLES / "test_per_etf.csv", index=False)

    seed_rows = []
    for s in SEEDS:
        a_, b_ = runs[(1.0, s)], runs[(ALPHA_STAR, s)]
        seed_rows.append(dict(seed=s, QLIKE_base=qlike_mean(a_["y_true"][test], a_["y_pred"][test]),
                              QLIKE_new=qlike_mean(b_["y_true"][test], b_["y_pred"][test])))
    seed_tab = pd.DataFrame(seed_rows)
    seed_tab["new_wins"] = seed_tab.QLIKE_new < seed_tab.QLIKE_base
    print(f"HR-ETE-GNN beats H-ETE-GNN on test QLIKE in {seed_tab.new_wins.sum()} of {len(seed_tab)} seeds")
    seed_tab.to_csv(TABLES / "test_per_seed.csv", index=False)

    fig, ax = plt.subplots(figsize=(12, 3.8))
    ax.plot(dt, np.cumsum(lb - ln), color="#1a1a2e", lw=1)
    ax.axhline(0, color="k", lw=.6)
    for d in changes[changes > dt[0]]: ax.axvline(d, color="#f39c12", lw=.4, alpha=.6)
    ax.set_title(f"Cumulative QLIKE advantage of {hf.arm_label(ALPHA_STAR)} over H-ETE-GNN (up = Rényi better)")
    plt.tight_layout(); plt.savefig(TABLES / "cumulative_qlike_advantage.png", dpi=150); plt.show()

## 7. What was written

All tables and figures are in `tables/`, every run's forecasts are in `runs/`, and the frozen data panel is `data/panel.pkl`. To redo a run, delete its `.npz` file and re-run its stage.

In [ ]:
for p in sorted(TABLES.iterdir()):
    print(p.relative_to(HERE))